# ☕ Cafe Sales Data Cleaning 

## Step 1 | Importing the Libraries

In [26]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual themes
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

## Step 2 | Loading the Raw Dataset

In [27]:
# Load raw dataset
cafe_records = pd.read_csv("dirty_cafe_sales.csv")
cafe_records.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## Step 3 | Inspecting the Data Structure

In [28]:
# Display dataset dimensions and summary info
print("Rows:", cafe_records.shape[0])
print("Cols:", cafe_records.shape[1])
cafe_records.info()

Rows: 10000
Cols: 8
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 1.1 MB


In [29]:
# Count null values in each column
cafe_records.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [30]:
# Count hidden invalid strings (ERROR and UNKNOWN)
bogus_labels = ["ERROR", "UNKNOWN"]
dirty_counts = cafe_records.apply(lambda col: col.isin(bogus_labels).sum())
dirty_counts[dirty_counts > 0]

Item                636
Quantity            341
Price Per Unit      354
Total Spent         329
Payment Method      599
Location            696
Transaction Date    301
dtype: int64

## Step 4 | Replacing 'ERROR' and 'UNKNOWN' with NaNs

In [31]:
# Replace dirty string placeholders with NaN
sales_ledger = cafe_records.replace(bogus_labels, np.nan)
sales_ledger.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

## Step 5 | Handling Missing Categorical Values

In [32]:
# Impute missing categorical values using column mode
top_item = sales_ledger["Item"].mode().iloc[0]
sales_ledger["Item"] = sales_ledger["Item"].fillna(top_item)

top_payment = sales_ledger["Payment Method"].mode().iloc[0]
sales_ledger["Payment Method"] = sales_ledger["Payment Method"].fillna(top_payment)

top_location = sales_ledger["Location"].mode().iloc[0]
sales_ledger["Location"] = sales_ledger["Location"].fillna(top_location)

print(f"Item filled with: {top_item}")
print(f"Payment filled with: {top_payment}")
print(f"Location filled with: {top_location}")

Item filled with: Juice
Payment filled with: Digital Wallet
Location filled with: Takeaway


## Step 6 | Handling Numeric Columns

In [33]:
# Convert numeric columns and fill missing values with median
sales_ledger["Quantity"] = pd.to_numeric(sales_ledger["Quantity"], errors="coerce")
sales_ledger["Price Per Unit"] = pd.to_numeric(sales_ledger["Price Per Unit"], errors="coerce")
sales_ledger["Total Spent"] = pd.to_numeric(sales_ledger["Total Spent"], errors="coerce")

median_qty = sales_ledger["Quantity"].median()
median_price = sales_ledger["Price Per Unit"].median()

sales_ledger["Quantity"] = sales_ledger["Quantity"].fillna(median_qty)
sales_ledger["Price Per Unit"] = sales_ledger["Price Per Unit"].fillna(median_price)

print(f"Quantity median: {median_qty}")
print(f"Price median: {median_price}")

Quantity median: 3.0
Price median: 3.0


## Step 7 | Recalculating the Total Spent Column

In [34]:
# Recalculate Total Spent = Quantity * Price Per Unit
sales_ledger["Total Spent"] = sales_ledger["Quantity"] * sales_ledger["Price Per Unit"]
sales_ledger[["Quantity", "Price Per Unit", "Total Spent"]].head(10)

,Quantity,Price Per Unit,Total Spent
0,2.0,2.0,4.0
1,4.0,3.0,12.0
2,4.0,1.0,4.0
3,2.0,5.0,10.0
4,2.0,2.0,4.0
5,5.0,4.0,20.0
6,3.0,3.0,9.0
7,4.0,4.0,16.0
8,5.0,3.0,15.0
9,5.0,4.0,20.0


## Step 8 | Parsing and Handling Transaction Date

In [35]:
# Parse dates (invalid values become NaT)
sales_ledger["Transaction Date"] = pd.to_datetime(sales_ledger["Transaction Date"], errors="coerce")
missing_dates = sales_ledger["Transaction Date"].isna().sum()
print(f"Missing dates: {missing_dates}")

Missing dates: 460


## Step 9 | Post-Cleaning Verification

In [36]:
# Verify remaining null values
sales_ledger.isnull().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [37]:
# View numeric columns summary statistics
sales_ledger.describe()

,Quantity,Price Per Unit,Total Spent,Transaction Date
count,10000.000000,10000.00000,10000.000000,9540
mean,3.027100,2.95265,8.948150,2023-07-01 23:00:31.698113
min,1.000000,1.00000,1.000000,2023-01-01 00:00:00
25%,2.000000,2.00000,4.000000,2023-04-01 00:00:00
50%,3.000000,3.00000,8.000000,2023-07-02 00:00:00
75%,4.000000,4.00000,12.000000,2023-10-02 00:00:00
max,5.000000,5.00000,25.000000,2023-12-31 00:00:00
std,1.384614,1.24396,5.831191,NaN


In [38]:
# Show first 10 rows of cleaned dataset
sales_ledger.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Takeaway,2023-03-31
6,TXN_4433211,Juice,3.0,3.0,9.0,Digital Wallet,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Takeaway,2023-10-28
8,TXN_4717867,Juice,5.0,3.0,15.0,Digital Wallet,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,Digital Wallet,In-store,2023-12-31


## Step 10 | Exporting the Cleaned Dataset

In [39]:
# Save clean dataset back to CSV
sales_ledger.to_csv("cleaned_cafe_sales.csv", index=False)
print("✅ Cleaned data exported successfully")
print(f"   Shape: {sales_ledger.shape}")

✅ Cleaned data exported successfully
   Shape: (10000, 8)
